# Main section

In [ ]:
# Imports
import numpy as np

a = np.zeros((4,4))
a

In [ ]:
import os 
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from datasets import Dataset


In [ ]:
from transformers import BertForSequenceClassification, AutoTokenizer
path = "google-bert/bert-base-cased"
if os.path.exists("tokenizer"):
    tokenizer = AutoTokenizer.from_pretrained("tokenizer")
else:
    tokenizer = AutoTokenizer.from_pretrained(path)    
    save_directory = "tokenizer"
    tokenizer.save_pretrained(save_directory)
if os.path.exists("model"):
    model = BertForSequenceClassification.from_pretrained("model")
else:
    model = BertForSequenceClassification.from_pretrained(path, num_labels=2)
    save_directory = "model"
    model.save_pretrained(save_directory)

# Preprocessing the dataset

In [ ]:
# Load the dataset
path = "dataset/fake_job_postings.csv"
dataset = pd.read_csv(path)

print(dataset.columns)

columns = dataset.columns
for col in columns:
    print(f"{col}: {dataset[col].isnull().sum()}")

print("Original shape:", dataset.shape)

# Drop columns with too many missing values (more than 1000 missing values)
no_empty = dataset.dropna(axis="columns",thresh=dataset.shape[0]-1000)
print("Shape after dropping columns with too many missing values:", no_empty.shape)

# Drop rows with missing values in the 'description' column
no_empty = no_empty.dropna(subset=['description'])
print('Dropped no of rows:', dataset.shape[0]-no_empty.shape[0])
columns = no_empty.columns
for col in columns:
    print(f"{col}: {no_empty[col].isnull().sum()}")

print(no_empty.head())


## visualization of the data

In [ ]:
org1, org2 = dataset.fraudulent.value_counts()
x1,x2 = no_empty.fraudulent.value_counts()
x3,x4 = no_empty.telecommuting.value_counts()
x5,x6 = no_empty.has_company_logo.value_counts()
x7,x8 = no_empty.has_questions.value_counts()
print(f"Not Fraudulent: {x1}, Fraudulent: {x2}")
print(f"Originally Not Fraudulent: {org1}, Originally Fraudulent: {org2}")
#plotting the distribution of fraudulent and non-fraudulent job postings

plt.figure(figsize=(10,5))
plt.bar(["Not Fraudulent", "Fraudulent", "No Telecommuting", "Telecommuting", "No Company Logo", "Company Logo", "No Questions", "Questions"], [x1, x2, x3, x4, x5, x6, x7, x8], color=["red", "green", "blue", "yellow", "orange", "purple", "pink", "brown"])
plt.title("Distribution of binary data in the job postings")
plt.xticks(rotation=15)
# fraudulent normalizeed 
x1,x2 = no_empty.fraudulent.value_counts(normalize=True)
plt.figure()
plt.bar(["Not Fraudulent", "Fraudulent"], [x1, x2])
plt.ylabel("Proportion")
plt.title("Proportion of fraudulent and non-fraudulent job postings")


In [ ]:

# more preprocessing for LLM input
# concatenate the text columns and binary columns into a single string for each row
text_columns = ['title', 'location', 'description']
binary_columns = ['telecommuting', 'has_company_logo', 'has_questions']

def build_data_for_llm(row):
    output = []
    for col in binary_columns:
        val = row[col]
        if val is None or pd.isna(val):
            val = "N/A"
        output.append(f"{col}: {'yes' if val == 1 else 'no'}")
    for col in text_columns:
        val = row[col]
        if val is None or pd.isna(val):
            val = "N/A"
        output.append(f"{col}: {val}")
        
    return " [SEP] ".join(output)

no_empty['text'] = no_empty.apply(build_data_for_llm, axis=1)
df = no_empty[["text", "fraudulent"]].rename(columns={"fraudulent": "label"})
print(df.head())

#seperate into train and test sets
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

In [ ]:
# Tokenize the text data
train_ds = Dataset.from_pandas(train_df)
train_ds = train_ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=512), batched=True)
val_ds = Dataset.from_pandas(val_df)
val_ds = val_ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=512), batched=True)
test_ds = Dataset.from_pandas(test_df)
test_ds = test_ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=512), batched=True)

In [ ]:
# check the size of the datasets
def count_tokens_of_each_row(ds):
    rows = ds.num_rows
    count = 0
    for item in ds:
        if len(item['input_ids']) < 512:
            count += 1
    return count

under_512_train = count_tokens_of_each_row(train_ds)
print(f"Number of rows in train_ds with less than 512 tokens: {under_512_train} out of {train_ds.num_rows}")
under_512_val = count_tokens_of_each_row(val_ds)
print(f"Number of rows in val_ds with less than 512 tokens: {under_512_val} out of {val_ds.num_rows}")
under_512_test = count_tokens_of_each_row(test_ds)
print(f"Number of rows in test_ds with less than 512 tokens: {under_512_test} out of {test_ds.num_rows}")
